# Full Day RFI Flagging on Sky-Filtered SNRs

**by Josh Dillon**, last updated September 21, 2026

This notebook brings together the night's [single-baseline sky-filtered
SNRs](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/phase_II/single_baseline_sky_filtered_SNR.ipynb),
averages their magnitudes incoherently over baselines into one z-score waterfall per polarization,
and flags whatever is not sky-like: single outliers and their neighbors, whole channels and
integrations, compact stretches in frequency, and whole digital TV allocations at a time ([HERA Memo
#82](https://reionization.org/wp-content/uploads/2013/03/HERA082_TV_Info.pdf)). It is patterned on
H6C's `full_day_rfi_round_3.ipynb`, and like it is iterative in the manner of
[full_day_rfi](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/phase_II/full_day_rfi.ipynb).
Its flags come on top of those already in the redundantly averaged data.

Configuration comes from a TOML file (e.g.
`hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml`) pointed to by the
`TOML_FILE` environment variable; without one, the defaults in the cells below are used. Other
environment variables carry only paths and wrapper-level toggles. `SUM_FILE` may be any raw file of
the night.

When `SAVE_RESULTS` is `TRUE`, one file is written for the night:

* `single_baseline_files/zen.{JD}.flag_waterfall_sky_filtered.h5` — a `UVFlag` waterfall of flags on
  the single-baseline files' whole-night time grid, the same for every polarization.

Here's a set of links to skip to particular figures:

• [Figure 1: Waterfall of Maximum z-Score of Either Polarization Before Flagging](#Figure-1:-Waterfall-of-Maximum-z-Score-of-Either-Polarization-Before-Flagging)

• [Figure 2: Histogram of z-Scores](#Figure-2:-Histogram-of-z-Scores)

• [Figure 3: Waterfall of Maximum z-Score of Either Polarization After Flagging](#Figure-3:-Waterfall-of-Maximum-z-Score-of-Either-Polarization-After-Flagging)

• [Figure 4: Summary of Flags Before and After This Round](#Figure-4:-Summary-of-Flags-Before-and-After-This-Round)

In [ ]:
import time
tstart = time.time()
!hostname
!date

In [ ]:
import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import h5py
import hdf5plugin  # REQUIRED to have the compression plugins available
import toml
import numpy as np
import yaml
import re
import warnings
import matplotlib
from scipy.signal import convolve2d
from pyuvdata import UVFlag
from hera_qm import xrfi
from hera_cal import io, flag_utils, utils
import matplotlib.pyplot as plt

from IPython.display import display, HTML
%matplotlib inline
display(HTML("<style>.container { width:100% !important; }</style>"))
_ = np.seterr(all='ignore')  # get rid of red warnings
%config InlineBackend.figure_format = 'retina'

## Parse inputs and outputs

To run interactively, provide a sum file path if the environment variable is not set; everything
else has defaults.

In [ ]:
# parse wrapper-level environment variables: paths, plus the save toggle
SUM_FILE = os.environ.get("SUM_FILE", None)
# SUM_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/2459935/zen.2459935.21341.sum.uvh5'
TOML_FILE = os.environ.get("TOML_FILE", None)
# TOML_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/src/hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml'
SAVE_RESULTS = os.environ.get("SAVE_RESULTS", "TRUE").upper() == "TRUE"
# which [WorkFlow] action is running this notebook
ACTION = os.environ.get("ACTION", "FULL_DAY_RFI_SKY_FILTERED_NOTEBOOK")

# default suffixes for the files this notebook reads; both end a single-baseline file's name
SINGLE_BASELINE_SUFFIX = 'sum.smooth_calibrated.red_avg.uvh5'
SKY_FILTERED_SNR_SUFFIX = 'sum.sky_filtered_SNR.uvh5'
CORNER_TURN_MAP_FILENAME = 'single_baseline_files/corner_turn_map.yaml'
FLAGS_SKY_FILTERED_FILENAME = 'single_baseline_files/zen.{JD}.flag_waterfall_sky_filtered.h5'

# shared across the pipeline via [GLOBAL_OPTS]
BAND_SPLIT_FREQ = 100.0  # in MHz; divides the low and high bands (inside the a priori flagged FM gap)
TV_CHAN_EDGES = [174, 182, 190, 198, 206, 214, 222, 230, 238, 246, 254]  # in MHz; edges of the digital TV allocations

# default settings, overridden by the TOML's [FULL_DAY_RFI_SKY_FILTERED_OPTS] section (if given)
MIN_SAMP_FRAC = 0.15  # baselines with fewer median nsamples than this fraction of the autocorrelations' are left out
Z_THRESH = 4.0
WS_Z_THRESH = 2.0
AVG_Z_THRESH = 1.0
MAX_FREQ_FLAG_FRAC = 0.25
MAX_TIME_FLAG_FRAC = 0.5
FREQ_CONV_SIZE = 8.0  # in MHz

toml_options = (toml.load(TOML_FILE) if TOML_FILE is not None else {})

# Take the suffixes this notebook reads or writes from [DATA_PRODUCTS], scoped by that
# section's produced_by/consumed_by wiring, and require that wiring to agree exactly with
# the defaults above. That way the arrows drawn on the pipeline flowchart cannot drift from
# what this notebook actually touches: adding an edge there without using the file here (or
# vice versa) fails loudly, in the first cell, rather than silently.
declared_suffixes = {name for name in list(globals()) if name.endswith('_SUFFIX')}
wired_suffixes = set()
for product, spec in toml_options.get('DATA_PRODUCTS', {}).items():
    producers, consumers = (spec.get(key, []) for key in ['produced_by', 'consumed_by'])
    producers = ([producers] if isinstance(producers, str) else producers)
    consumers = ([consumers] if isinstance(consumers, str) else consumers)
    if 'suffix' in spec and (ACTION in producers or ACTION in consumers):
        wired_suffixes.add(f'{product.upper()}_SUFFIX')
        globals()[f'{product.upper()}_SUFFIX'] = spec['suffix']
if toml_options:
    assert wired_suffixes == declared_suffixes, (
        f'[DATA_PRODUCTS] wires {sorted(wired_suffixes)} to {ACTION}, '
        f'but this notebook declares {sorted(declared_suffixes)}.')
# per-night products have no per-file suffix; their [DATA_PRODUCTS] filenames are relative to SUM_FILE's folder
for product in [name[:-len('_FILENAME')] for name in list(globals()) if name.endswith('_FILENAME')]:
    globals()[f'{product}_FILENAME'] = toml_options.get('DATA_PRODUCTS', {}).get(product, {}).get('filename', globals()[f'{product}_FILENAME'])

for toml_section in ['GLOBAL_OPTS', 'FULL_DAY_RFI_SKY_FILTERED_OPTS']:
    if toml_section in toml_options:
        print(f'Loading overrides from [{toml_section}] in {TOML_FILE}.')
        for key, val in toml_options[toml_section].items():
            globals()[key.upper()] = val

CORNER_TURN_MAP_YAML = os.path.join(os.path.dirname(SUM_FILE), CORNER_TURN_MAP_FILENAME)
OUTFILE = os.path.join(os.path.dirname(SUM_FILE), FLAGS_SKY_FILTERED_FILENAME.format(JD=re.search(r'zen\.(\d+)\.', os.path.basename(SUM_FILE)).group(1)))

for setting in ['SUM_FILE', 'TOML_FILE', 'SAVE_RESULTS', 'ACTION', 'SINGLE_BASELINE_SUFFIX', 'SKY_FILTERED_SNR_SUFFIX',
                'CORNER_TURN_MAP_YAML', 'OUTFILE', 'BAND_SPLIT_FREQ', 'TV_CHAN_EDGES', 'MIN_SAMP_FRAC', 'Z_THRESH', 'WS_Z_THRESH',
                'AVG_Z_THRESH', 'MAX_FREQ_FLAG_FRAC', 'MAX_TIME_FLAG_FRAC', 'FREQ_CONV_SIZE']:
    print(f'{setting} = {eval(setting)}')

## Load the sky-filtered SNRs and turn them into z-scores

In [ ]:
with open(CORNER_TURN_MAP_YAML, 'r') as file:
    corner_turn_map = yaml.unsafe_load(file)

all_outfiles = [outfile for outfiles in corner_turn_map['files_to_outfiles_map'].values() for outfile in outfiles]
all_snr_files = [outfile[:-len(SINGLE_BASELINE_SUFFIX)] + SKY_FILTERED_SNR_SUFFIX for outfile in all_outfiles]
extant_snr_files = [snr_file for snr_file in all_snr_files if os.path.exists(snr_file)]
print(f'Found {len(extant_snr_files)} SNR files, starting with {extant_snr_files[0]}')

In [ ]:
# get autocorrelations, for the whole-night time and frequency grid and for their nsamples
for outfile in all_outfiles:
    match = re.search(r'\.(\d+)_(\d+)\.', os.path.basename(outfile))
    if match and match.group(1) == match.group(2):
        hd_autos = io.HERAData(outfile)
        _, _, auto_nsamples = hd_autos.read(polarizations=['ee', 'nn'])
        break
times, freqs = hd_autos.times, hd_autos.freqs
med_auto_nsamples = {bl[2]: np.median(n) for bl, n in auto_nsamples.items()}

In [ ]:
# define slices for TV allocations
tv_slices = []
for low_edge, high_edge in zip(TV_CHAN_EDGES[:-1], TV_CHAN_EDGES[1:]):
    chans_in_band = np.argwhere((freqs / 1e6 > low_edge) & (freqs / 1e6 < high_edge))
    if len(chans_in_band) > 0:
        tv_slices.append(slice(np.min(chans_in_band), np.max(chans_in_band) + 1))

In [ ]:
# combine |SNR|s incoherently as they are loaded, leaving out baselines with too few samples
abs_SNR_sum = {pol: np.zeros((len(times), len(freqs)), dtype=float) for pol in ['ee', 'nn']}
abs_SNR_count = {pol: np.zeros((len(times), len(freqs)), dtype=float) for pol in ['ee', 'nn']}
bls_used = []
for snr_file in extant_snr_files:
    hd = io.HERADataFastReader(snr_file)
    data, flags, nsamples = hd.read()
    for bl in data:
        if (np.median(nsamples[bl][~flags[bl]]) > MIN_SAMP_FRAC * med_auto_nsamples[bl[2]]) and (np.linalg.norm(hd.antpos[bl[0]] - hd.antpos[bl[1]]) > 1):
            invalid = flags[bl] | ~np.isfinite(data[bl])
            bls_used.append(bl)
            abs_SNR_sum[bl[2]] += np.where(invalid, 0, np.abs(data[bl]))
            abs_SNR_count[bl[2]] += ~invalid
print(f'Combined {len(bls_used)} baseline-polarizations.')

In [ ]:
# convert SNRs to a z-score
zscore = {}
for pol in abs_SNR_sum.keys():
    predicted_mean = 1.0
    sigma = predicted_mean * np.sqrt(2 / np.pi)
    variance_expected = (4 - np.pi) / 2 * sigma**2 / abs_SNR_count[pol]
    zscore[pol] = (abs_SNR_sum[pol] / abs_SNR_count[pol] - predicted_mean) / variance_expected**.5
    zscore[pol] = np.where(abs_SNR_count[pol] == 0, np.nan, zscore[pol])

In [ ]:
# recenter z-scores above and below BAND_SPLIT_FREQ and per-polarization
_, (low_band, high_band) = flag_utils.get_minimal_slices(np.any(~np.isfinite(list(zscore.values())), axis=0),
                                                         freqs=freqs, freq_cuts=[BAND_SPLIT_FREQ * 1e6])
for pol in zscore:
    for band in [low_band, high_band]:
        zscore[pol][:, band] -= np.nanmedian(zscore[pol][:, band])

## Plotting Functions

In [ ]:
lst_grid = utils.JD2LST(times) * 12 / np.pi
lst_grid[lst_grid > lst_grid[-1]] -= 24

def plot_max_z_score(zscore, flags=None, vmin=-5, vmax=5):
    if flags is None:
        flags = np.any(~np.isfinite(list(zscore.values())), axis=0)
    fig, ax = plt.subplots(figsize=(14, 10), dpi=150)
    extent = [freqs[0] / 1e6, freqs[-1] / 1e6, 
              times[-1] - int(times[0]), times[0] - int(times[0])]
    
    im = ax.imshow(np.where(flags, np.nan, np.nanmax([zscore['ee'], zscore['nn']], axis=0)), aspect='auto', 
               cmap='coolwarm', interpolation='none', vmin=vmin, vmax=vmax, extent=extent)
    plt.colorbar(im, ax=ax, location='top', label='Max z-score of either polarization', extend='both', aspect=40, pad=.02)
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel(f'JD - {int(times[0])}')

    # Add LST right axis
    ax2 = ax.twinx()
    ax2.set_ylim(lst_grid[-1], lst_grid[0])
    mod24 = lambda x, _: f"{x % 24:.1f}"
    ax2.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
    ax2.set_ylabel('LST (hours)')

    plt.tight_layout()
    for freq in TV_CHAN_EDGES:
        if freq < freqs[-1] * 1e-6:
            ax.axvline(freq, lw=.5, ls='--', color='k')

In [ ]:
def plot_histogram():
    plt.figure(figsize=(14,4), dpi=100)
    bins = np.arange(-50, 100, .1)
    hist_ee = plt.hist(np.ravel(zscore['ee']), bins=bins, density=True, label='ee-polarized z-scores', alpha=.5)
    hist_nn = plt.hist(np.ravel(zscore['nn']), bins=bins, density=True, label='nn-polarized z-scores', alpha=.5)
    plt.plot(bins, (2*np.pi)**-.5 * np.exp(-bins**2 / 2), 'k:', label='Gaussian approximate\nnoise-only distribution')
    plt.axvline(WS_Z_THRESH, c='r', ls='--', label='Watershed z-score')
    plt.axvline(Z_THRESH, c='r', ls='-', label='Threshold z-score')    
    plt.yscale('log')
    all_densities = np.concatenate([hist_ee[0][hist_ee[0] > 0], hist_nn[0][hist_nn[0] > 0]]) 
    plt.ylim(np.min(all_densities) / 2, np.max(all_densities) * 2)
    plt.xlim([-50, 100])
    plt.legend()
    plt.xlabel('z-score')
    plt.ylabel('Density')
    plt.tight_layout()

In [ ]:
def summarize_flagging(flags):
    fig, ax = plt.subplots(figsize=(14, 10), dpi=150)
    cmap = matplotlib.colors.ListedColormap(((0, 0, 0),) + matplotlib.cm.get_cmap("Set2").colors[0:2])
    extent = [freqs[0] / 1e6, freqs[-1] / 1e6, 
              times[-1] - int(times[0]), times[0] - int(times[0])]    
    im = ax.imshow(np.where(np.any(~np.isfinite(list(zscore.values())), axis=0), 1, np.where(flags, 2, 0)), 
               aspect='auto', cmap=cmap, interpolation='none', extent=extent)
    im.set_clim([-.5, 2.5])
    cbar = plt.colorbar(im, ax=ax, location='top', aspect=40, pad=.02)
    cbar.set_ticks([0, 1, 2])
    cbar.set_ticklabels(['Unflagged', 'Flagged Before This Round', 'Flagged by This Round'])
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel(f'JD - {int(times[0])}')

    # Add LST right axis
    ax2 = ax.twinx()
    ax2.set_ylim(lst_grid[-1], lst_grid[0])
    mod24 = lambda x, _: f"{x % 24:.1f}"
    ax2.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
    ax2.set_ylabel('LST (hours)')

    plt.tight_layout()

# *Figure 1: Waterfall of Maximum z-Score of Either Polarization Before Flagging*

The worse (higher z-score) of the two polarizations. Dashed lines in the high band mark the digital
TV allocations, which receive special treatment. Large positive excursions are a problem and likely
need flagging. The bands above and below `BAND_SPLIT_FREQ` are handled separately and may have
different levels of post-filtering.

In [ ]:
plot_max_z_score(zscore)

# *Figure 2: Histogram of z-Scores*

The histogram of z-scores against a Gaussian approximation of what thermal noise would give. Without
filtering, the actual distribution is a weighted sum of Rayleigh distributions, and filtering
complicates it further, so a single median per polarization and band is subtracted from each
waterfall to make low-level outliers flaggable with more confidence. Points beyond the solid red
line are flagged, as are points neighboring a flag beyond the dashed red line. Finally, low-level
outliers are flagged over whole times or channels, in TV allocations, and in other compact stretches
of frequency.

In [ ]:
plot_histogram()

## Flagging functions

In [ ]:
def iteratively_flag_on_averaged_zscore(flags, zscore, avg_func=np.nanmean, avg_z_thresh=AVG_Z_THRESH, verbose=True):
    '''Flag whole integrations or channels based on average z-score. This is done
    iteratively to prevent bad times affecting channel averages or vice versa.'''

    _, (low_band, high_band) = flag_utils.get_minimal_slices(flags, freqs=freqs, freq_cuts=[BAND_SPLIT_FREQ * 1e6])
    flagged_chan_count = 0
    flagged_int_count = {low_band: 0, high_band: 0}
    for band in (low_band, high_band):
        while True:
            zspec = avg_func(np.where(flags, np.nan, zscore)[:, band], axis=0)
            ztseries = avg_func(np.where(flags, np.nan, zscore)[:, band], axis=1)
    
            if (np.nanmax(zspec) < avg_z_thresh) and (np.nanmax(ztseries) < avg_z_thresh):
                break
    
            if np.nanmax(zspec) >= np.nanmax(ztseries):
                flagged_chan_count += np.sum((zspec >= np.nanmax(ztseries)) & (zspec >= avg_z_thresh))
                flags[:, band][:, (zspec >= np.nanmax(ztseries)) & (zspec >= avg_z_thresh)] = True
            else:
                flagged_int_count[band] += np.sum((ztseries >= np.nanmax(zspec)) & (ztseries >= avg_z_thresh))
                flags[(ztseries >= np.nanmax(zspec)) & (ztseries >= avg_z_thresh), band] = True

    ztseries_low = avg_func(np.where(flags, np.nan, zscore)[:, low_band], axis=1)
    flags[(ztseries_low > avg_z_thresh) & np.all(flags[:, high_band], axis=1), low_band] = True
    
    if verbose:
        if (flagged_int_count[low_band] > 0) or (flagged_int_count[high_band] > 0) or (flagged_chan_count > 0):
            print(f'\tFlagging an additional {flagged_int_count[low_band]} low-band integrations, '
                  f'{flagged_int_count[high_band]} high-band integrations, and {flagged_chan_count} channels.')

def impose_max_chan_flag_frac(flags, max_flag_frac=MAX_FREQ_FLAG_FRAC, verbose=True):
    '''Flag channels already flagged more than max_flag_frac (excluding completely flagged times).'''
    _, (low_band, high_band) = flag_utils.get_minimal_slices(flags, freqs=freqs, freq_cuts=[BAND_SPLIT_FREQ * 1e6])
    for band in [low_band, high_band]:
        unflagged_times = ~np.all(flags[:, band], axis=1)
        frequently_flagged_chans =  np.mean(flags[unflagged_times, band], axis=0) >= max_flag_frac
        if verbose:
            flag_diff_count = np.sum(frequently_flagged_chans) - np.sum(np.all(flags[:, band], axis=0))
            if flag_diff_count > 0:
                print(f'\tFlagging {flag_diff_count} channels previously flagged {max_flag_frac:.2%} or more.')        
        flags[:, band][:, frequently_flagged_chans] = True
        
def impose_max_time_flag_frac(flags, max_flag_frac=MAX_TIME_FLAG_FRAC, verbose=True):
    '''Flag times already flagged more than max_flag_frac (excluding completely flagged channels).'''
    _, (low_band, high_band) = flag_utils.get_minimal_slices(flags, freqs=freqs, freq_cuts=[BAND_SPLIT_FREQ * 1e6])
    for name, band in zip(['low', 'high'], [low_band, high_band]):
        unflagged_chans = ~np.all(flags[:, band], axis=0)
        frequently_flagged_times =  np.mean(flags[:, band][:, unflagged_chans], axis=1) >= max_flag_frac
        if verbose:
            flag_diff_count = np.sum(frequently_flagged_times) - np.sum(np.all(flags[:, band], axis=1))
            if flag_diff_count > 0:
                print(f'\tFlagging {flag_diff_count} {name}-band times previously flagged {max_flag_frac:.2%} or more.')
        flags[frequently_flagged_times, band] = True

def flag_tv(flags, zscore, tv_thresh=AVG_Z_THRESH, egregious_thresh=(2 * Z_THRESH)):
    '''Flag single-time TV allocations with average zscores above tv_thresh, excluding particularly bad channels.'''
    for pol in zscore:
        for tvs in tv_slices:
            for tind in range(zscore[pol].shape[0]):
                if np.nanmean(zscore[pol][tind, tvs]) > tv_thresh:
                    egregious_outliers = zscore[pol][tind, tvs] > egregious_thresh
                    if np.nanmean(zscore[pol][tind, tvs][~egregious_outliers]) > tv_thresh:
                        # even without the worst outliers, it still looks like there's TV in this whole allocation
                        flags[tind, tvs] = True
                    else:
                        # just flag the most egregious outliers
                        flags[tind, tvs] |= egregious_outliers

def watershed_flag(flags, zscore, ws_z_thresh=WS_Z_THRESH):
    '''Wrapper around xrfi._ws_flag_waterfall to be performed separately above and below FM.'''
    while True:        
        nflags = np.sum(flags)
        _, (low_band, high_band) = flag_utils.get_minimal_slices(flags, freqs=freqs, freq_cuts=[BAND_SPLIT_FREQ * 1e6])
        for band in [low_band, high_band]:
            for pol in ['ee', 'nn']:
                flags[:, band] |= xrfi._ws_flag_waterfall(zscore[pol][:, band], flags[:, band], ws_z_thresh)
        if np.sum(flags) == nflags:
            break

def iterative_freq_conv_flagging(flags, zscore, conv_size=FREQ_CONV_SIZE, one_chan_thresh=Z_THRESH, full_kernel_thresh=AVG_Z_THRESH):
    '''Looks for streteches of increasing size that fit a decreasing threshold. At conv_size (in MHz), it flags 
    stretches with average z-score above full_kernel_thresh. At one pixel, it uses one_chan_thresh.
    In between, it interpolates logarithmically.'''
    _, (low_band, high_band) = flag_utils.get_minimal_slices(flags, freqs=freqs, freq_cuts=[BAND_SPLIT_FREQ * 1e6])
    df_MHz = np.median(np.diff(freqs)) / 1e6
    widths = np.array([int(w) + 1 for w in 2**np.arange(1, np.ceil(np.log2(conv_size / df_MHz) + np.finfo(float).eps))])
    
    # prevent any widths from being so big that they mix high and low bands
    max_width = (high_band.start - low_band.stop) * 2
    widths[widths > max_width] = max_width
    widths = np.unique(widths)

    # Create cuts that get more strict as the kernel gets bigger
    cuts = one_chan_thresh * (full_kernel_thresh / one_chan_thresh)**((widths - 1) / (conv_size / df_MHz - 1))

    for width, cut in zip(widths, cuts):
        result = {}
        for pol in zscore.keys():
            kernel = np.ones((1, int(width)), dtype=float)
            mask = ~(np.isnan(zscore[pol]) | flags)
            filled_data = np.where(mask, zscore[pol], 0.0)
            conv_data = convolve2d(filled_data, kernel, mode='same')
            conv_mask = convolve2d(mask.astype(float), kernel, mode='same')
            with np.errstate(divide='ignore', invalid='ignore'):
                result[pol] = conv_data / conv_mask

        for band in [low_band, high_band]:
            above_cut = np.any([result[pol][:, band] > cut for pol in result.keys()], axis=0)
            flags[:, band] |= (convolve2d(above_cut.astype(float), kernel, mode='same') > 0)
        
        print(f'{np.mean(flags):.3%} of waterfall flagged after {width}-channel convolution-based flagging with z-scores above {cut:.3f}.')

## Main Flagging Routine

In [ ]:
flags = np.any(~np.isfinite(list(zscore.values())), axis=0)
_, (low_band, high_band) = flag_utils.get_minimal_slices(flags, freqs=freqs, freq_cuts=[BAND_SPLIT_FREQ * 1e6])
print(f'{np.mean(flags):.3%} of waterfall flagged to start.')

# flag bad TV allocations
flag_tv(flags, zscore)
print(f'{np.mean(flags):.3%} of waterfall flagged after TV channel cuts.')

# flag whole integrations or channels using outliers in median
while True:
    nflags = np.sum(flags)
    for pol in ['ee', 'nn']:    
        iteratively_flag_on_averaged_zscore(flags, zscore[pol], avg_func=np.nanmedian, avg_z_thresh=AVG_Z_THRESH, verbose=True)
        impose_max_chan_flag_frac(flags, max_flag_frac=MAX_FREQ_FLAG_FRAC, verbose=True)
        impose_max_time_flag_frac(flags, max_flag_frac=MAX_TIME_FLAG_FRAC, verbose=True)
    if np.sum(flags) == nflags:
        break  
print(f'{np.mean(flags):.3%} of waterfall flagged after flagging whole times and channels with median z > {AVG_Z_THRESH}.')

# flag largest outliers
_, (low_band, high_band) = flag_utils.get_minimal_slices(flags, freqs=freqs, freq_cuts=[BAND_SPLIT_FREQ * 1e6])
for band in [low_band, high_band]:
    for pol in ['ee', 'nn']:
        flags[:, band] |= (zscore[pol][:, band] > Z_THRESH) 
print(f'{np.mean(flags):.3%} of waterfall flagged after flagging z > {Z_THRESH} outliers.')

# watershed flagging
watershed_flag(flags, zscore, ws_z_thresh=WS_Z_THRESH)
print(f'{np.mean(flags):.3%} of waterfall flagged after watershed flagging on z > {WS_Z_THRESH} neighbors of prior flags.')

# iterative frequency-convolved flagging
iterative_freq_conv_flagging(flags, zscore, conv_size=FREQ_CONV_SIZE, one_chan_thresh=Z_THRESH, full_kernel_thresh=AVG_Z_THRESH)
print(f'{np.mean(flags):.3%} of waterfall flagged after channel convolution flagging.')

# watershed flagging
watershed_flag(flags, zscore, ws_z_thresh=WS_Z_THRESH)
print(f'{np.mean(flags):.3%} of waterfall flagged after watershed flagging again on z > {WS_Z_THRESH} neighbors of prior flags.')

# flag whole integrations or channels using outliers in mean
while True:
    nflags = np.sum(flags)
    for pol in ['ee', 'nn']:    
        iteratively_flag_on_averaged_zscore(flags, zscore[pol], avg_func=np.nanmean, avg_z_thresh=AVG_Z_THRESH, verbose=True)
        impose_max_chan_flag_frac(flags, max_flag_frac=MAX_FREQ_FLAG_FRAC, verbose=True)
        impose_max_time_flag_frac(flags, max_flag_frac=MAX_TIME_FLAG_FRAC, verbose=True)
    if np.sum(flags) == nflags:
        break  
print(f'{np.mean(flags):.3%} of waterfall flagged after flagging whole times and channels with average z > {AVG_Z_THRESH}.')

# watershed flagging again
watershed_flag(flags, zscore, ws_z_thresh=WS_Z_THRESH)
print(f'{np.mean(flags):.3%} of waterfall flagged after watershed flagging one last time on z > {WS_Z_THRESH} neighbors of prior flags.')

# *Figure 3: Waterfall of Maximum z-Score of Either Polarization After Flagging*

Same as [Figure 1](#Figure-1:-Waterfall-of-Maximum-z-Score-of-Either-Polarization-Before-Flagging)
above, but now with this round's flags.

In [ ]:
plot_max_z_score(zscore, flags=flags)

# *Figure 4: Summary of Flags Before and After This Round*

Which times and frequencies were flagged before and by this notebook. It is directly comparable to
the flag summary of the first-round
[full_day_rfi](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/phase_II/full_day_rfi.ipynb)
notebook.

In [ ]:
summarize_flagging(flags)

## Save results

In [ ]:
if SAVE_RESULTS:
    uvf = UVFlag(hd_autos, mode='flag', waterfall=True)
    for polind in range(uvf.flag_array.shape[2]):
        uvf.flag_array[:, :, polind] = flags
    uvf.write(OUTFILE, clobber=True)
    print(f'Wrote {OUTFILE}')

## Metadata

In [ ]:
for repo in ['hera_cal', 'hera_qm', 'hera_filters', 'hera_notebook_templates', 'pyuvdata']:
    exec(f'from {repo} import __version__')
    print(f'{repo}: {__version__}')

In [ ]:
print(f'Finished execution in {(time.time() - tstart) / 60:.2f} minutes.')